In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path

project_root = Path.cwd().parent
model_dir = project_root / "contract-category-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 524.59it/s]


In [8]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

from src.chunking import get_chunks

In [4]:
test_chunks = get_chunks(project_root / "data/test.json", tokenizer, relevant_categories)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2413 > 512). Running this sequence through the model will result in indexing errors


In [6]:
test_chunks.head()

,contract,paragraph_index,text,start_token,end_token,char_start,char_end,num_tokens,categories,partial_categories,label_status
0,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...,0,439,0,2042,439,[None],[],positive
1,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,2.3 If the seller puts forward amendments or n...,356,789,1658,3647,433,[None],[],negative
2,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,SHIPPING MARK: The Sellers shall mark on each ...,671,1116,3174,5156,445,[None],[],positive
3,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,(2) One certificate of origin of the goods. (3...,1040,1443,4792,6656,403,[None],[],negative
4,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,SHIPPING ADVICE: The seller shall within 72 ho...,1349,1761,6190,8189,412,[None],[],positive


In [12]:
MINIMUM_RECALL = {
    "Cap On Liability": 0.90,
    "Termination For Convenience": 0.85,
    "Governing Law": 0.80,
    "Anti-Assignment": 0.80,
    "Change Of Control": 0.85,
    "Renewal Term": 0.80,
    "Exclusivity": 0.85,
    "Non-Compete": 0.85,
    "Notice Period To Terminate Renewal": 0.80,
    "Liquidated Damages": 0.90,
}

MINIMUM_RECALL = {k.casefold(): v for k, v in MINIMUM_RECALL.items()}

In [13]:
import numpy as np
from sklearn.metrics import precision_recall_curve


def find_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_prob,
    )

    # precision and recall contain one more element than thresholds.
    precision = precision[:-1]
    recall = recall[:-1]

    f1 = (
        2 * precision * recall
        / np.maximum(precision + recall, 1e-12)
    )

    best_index = np.argmax(f1)

    return {
        "threshold": float(thresholds[best_index]),
        "precision": float(precision[best_index]),
        "recall": float(recall[best_index]),
        "f1": float(f1[best_index]),
    }

def find_recall_constrained_threshold(
    y_true,
    y_prob,
    minimum_recall=0.80,
):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_prob,
    )

    precision = precision[:-1]
    recall = recall[:-1]

    valid_indices = np.where(recall >= minimum_recall)[0]

    if len(valid_indices) == 0:
        return find_f1_threshold(y_true, y_prob)

    # Among thresholds meeting the recall requirement,
    # choose the one with the highest precision.
    best_index = valid_indices[
        np.argmax(precision[valid_indices])
    ]

    f1 = (
        2 * precision[best_index] * recall[best_index]
        / max(
            precision[best_index] + recall[best_index],
            1e-12,
        )
    )

    return {
        "threshold": float(thresholds[best_index]),
        "precision": float(precision[best_index]),
        "recall": float(recall[best_index]),
        "f1": float(f1),
    }

In [14]:
def tune_thresholds_with_fallback(
    y_true,
    y_prob,
    categories,
    minimum_recall_by_category,
    fallback_threshold=0.50,
    minimum_positives=10,
):
    results = {}

    for index, category in enumerate(categories):
        category_true = y_true[:, index]
        category_prob = y_prob[:, index]
        positive_count = int(category_true.sum())

        if positive_count < minimum_positives:
            results[category] = {
                "threshold": fallback_threshold,
                "positive_count": positive_count,
                "method": "fallback",
            }
            continue

        result = find_recall_constrained_threshold(
            y_true=category_true,
            y_prob=category_prob,
            minimum_recall=minimum_recall_by_category[category],
        )

        result["positive_count"] = positive_count
        result["method"] = "recall_constrained"
        results[category] = result

    return results

In [15]:
import numpy as np
import torch

val_chunks = get_chunks(project_root / "data/val.json", tokenizer, relevant_categories)

texts = val_chunks["text"].fillna("").astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 8

batch_probabilities = []

for start in range(0, len(texts), batch_size):
    batch_texts = texts[start:start + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.sigmoid(logits)

    batch_probabilities.append(probabilities.cpu().numpy())

y_prob = np.vstack(batch_probabilities)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (869 > 512). Running this sequence through the model will result in indexing errors


In [16]:
# Use the same label order as y_prob
label_names = [
    model.config.id2label[i]
    for i in range(model.config.num_labels)
]

label_to_index = {
    label: index
    for index, label in enumerate(label_names)
}

y_true = np.zeros(
    (len(val_chunks), len(label_names)),
    dtype=np.int64,
)

for row, categories in enumerate(val_chunks["categories"]):
    for label in categories:
        if label in label_to_index:
            y_true[row, label_to_index[label]] = 1

In [17]:
global_result = find_recall_constrained_threshold(
    y_true=y_true.ravel(),
    y_prob=y_prob.ravel()
)

global_threshold = global_result["threshold"]

In [18]:
threshold_results = tune_thresholds_with_fallback(
    y_true=y_true,
    y_prob=y_prob,
    categories=relevant_categories,
    minimum_recall_by_category=MINIMUM_RECALL,
    fallback_threshold=global_threshold,
    minimum_positives=10
)

CATEGORY_THRESHOLDS = {
    category: result["threshold"]
    for category, result in threshold_results.items()
}

In [ ]:
CATEGORY_THRESHOLDS
'''
{'cap on liability': 0.030137067660689354,
 'termination for convenience': 0.016307096928358078,
 'governing law': 0.9621155858039856,
 'anti-assignment': 0.777833104133606,
 'change of control': 0.048025332391262054,
 'renewal term': 0.657200038433075,
 'exclusivity': 0.04591374099254608,
 'non-compete': 0.001507436390966177,
 'notice period to terminate renewal': 0.4968838691711426,
 'liquidated damages': 0.18002332746982574}
'''

{'cap on liability': 0.030137067660689354,
 'termination for convenience': 0.016307096928358078,
 'governing law': 0.9621155858039856,
 'anti-assignment': 0.777833104133606,
 'change of control': 0.048025332391262054,
 'renewal term': 0.657200038433075,
 'exclusivity': 0.04591374099254608,
 'non-compete': 0.001507436390966177,
 'notice period to terminate renewal': 0.4968838691711426,
 'liquidated damages': 0.18002332746982574}

In [23]:
import torch

def predict(model, tokenizer, chunks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    labels = model.config.id2label  # Maps output index to category name

    batch_size = 8
    results = []

    texts = chunks["text"].fillna("").astype(str).tolist()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probabilities = torch.sigmoid(logits)

        for i, scores in enumerate(probabilities.cpu(), start=start):
            preds = {
                labels[i]: score.item()
                for i, score in enumerate(scores)
            }

            predicted_categories = [(category, score) for category, score in preds.items() 
                                    if score >= CATEGORY_THRESHOLDS.get(category.casefold(), threshold)]

            result = {
                "contract_id": chunks["contract_id"].iloc[i],
                "evidence_start": int(chunks["char_start"].iloc[i]),
                "evidence_end": int(chunks["char_end"].iloc[i]),
                "text": chunks["text"].iloc[i],
                "predicted_categories": [v[0] for v in predicted_categories],
                "predicted_scores": [v[1] for v in predicted_categories]
            }
            results.append(result)

    return results

In [21]:
import torch

def evaluate(model, tokenizer, texts):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    labels = model.config.id2label  # Maps output index to category name

    batch_size = 8
    results = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probabilities = torch.sigmoid(logits)

        for scores in probabilities.cpu():
            results.append({
                labels[i]: score.item()
                for i, score in enumerate(scores)
            })
    return results

In [22]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

# Use the same label order that was used when training.
labels = list(relevant_categories)

texts = test_chunks["text"].fillna("").astype(str).tolist()

results = evaluate(model, tokenizer, texts)

predicted_categories = [
    [category for category, score in row.items() if score >= CATEGORY_THRESHOLDS.get(category.casefold(), 0.5)]
    for row in results
]

# Convert true and predicted category lists into matching binary matrices.
true_label_lists = [
    [label for label in (categories or []) if label in labels]
    for categories in test_chunks["categories"]
]

mlb = MultiLabelBinarizer(classes=labels)
y_true = mlb.fit_transform(true_label_lists)
y_pred = mlb.transform(predicted_categories)

print("Micro F1:", f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))

Micro F1: 0.4212567882079131
Macro F1: 0.5805429519141667


In [ ]:
import json

data = predict(model, tokenizer, test_chunks)
def save_predictions_jsonl(predictions, path):
    with open(path, "w", encoding="utf-8") as file:
        for prediction in predictions:
            file.write(json.dumps(prediction, ensure_ascii=False) + "\n")


save_predictions_jsonl(data, "predictions.jsonl")